# UC CosMx: 1K+WTX QC, Normalization, Annotation, and Imputation Comparison (Fig 2, Ext Fig 6G)

1. QC and normalization of 1K + WTX combined panels
2. scRNA annotation comparison (1K vs 6K vs imputed)
3. YWL-B v7 GAE imputation annotation
4. Negative control validation (Ext Fig 6G): GCN imputed expression post-processing

In [ ]:
# ── Paths ──
DATA_DIR   = "/path/to/cosmx_data/uc_1k_6k_wtx"
K1_DIR     = DATA_DIR + "/1k/Processed_merged"
WTX_DIR    = DATA_DIR + "/whole_trans/Processed_merged"
K6_DIR     = DATA_DIR + "/6k/Processed_merged_upd"
SCRNA_DIR  = "/path/to/scrna/uc"
OUTPUT_DIR = K1_DIR

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [ ]:
sample_dir = '/path/to/cosmx_data/uc_1k_6k_wtx/whole_trans/3 Raw data/R6061_UC_P1_UC_P2/'

In [ ]:
fov_file = [item for item in os.listdir(sample_dir) if 'fov_positions_file' in item][0]
fov_df = pd.read_csv(os.path.join(sample_dir, fov_file))
if 'FOV' in fov_df.columns:
  print("Refactoring file to older format.")
  # Rename 'FOV' column to 'fov'
  fov_df.rename(columns={'FOV': 'fov'}, inplace=True)
  # have fov_file reference the new, formatted file and write it
  fov_file = os.path.join(sample_dir,'fov_positions_formatted.csv')
  fov_df.to_csv(fov_file, index=False)

In [ ]:
adata = sq.read.nanostring(
    path=sample_dir,
    counts_file="wtx_UC_P1_UC_P2_exprMat_file.csv.gz",
    meta_file="wtx_UC_P1_UC_P2_metadata_file.csv.gz",
    fov_file="fov_positions_formatted.csv",
)


In [ ]:
sample_dir = '/path/to/cosmx_data/uc_1k_6k_wtx/whole_trans/3 Raw data/R6061_UC_P3_UC_P4/'



In [ ]:
fov_file = [item for item in os.listdir(sample_dir) if 'fov_positions_file' in item][0]
fov_df = pd.read_csv(os.path.join(sample_dir, fov_file))
if 'FOV' in fov_df.columns:
  print("Refactoring file to older format.")
  # Rename 'FOV' column to 'fov'
  fov_df.rename(columns={'FOV': 'fov'}, inplace=True)
  # have fov_file reference the new, formatted file and write it
  fov_file = os.path.join(sample_dir,'fov_positions_formatted.csv')
  fov_df.to_csv(fov_file, index=False)

In [ ]:
adata2 = sq.read.nanostring(
    path=sample_dir,
    counts_file="wtx_UC_P3_UC_P4_exprMat_file.csv.gz",
    meta_file="wtx_UC_P3_UC_P4_metadata_file.csv.gz",
    fov_file="fov_positions_formatted.csv",
)


In [ ]:
adata.var['features']=adata.var.index.tolist()
adata.obs[['orig.ident']]='UC_batch1'
adata.obs['patient']=['UC_P2' if int(i) <=31 else 'UC_P1' for i in adata.obs['fov'].tolist() ]
adata.obs['condition']= 'Inflamed'
adata.obs['fov_cell_id']= [str(i)+'_'+str(j) for i,j in zip(adata.obs['fov'].tolist(), adata.obs['cell_id'].tolist())]

In [ ]:
adata.var['neg_probe']=[True if 'Negative' in i else False for i in adata.var.index.tolist()]
adata.var['control_prob']=[True if 'SystemContro' in i else False for i in adata.var.index.tolist()]
adata.var

In [ ]:
adata2.var['features']=adata2.var.index.tolist()
adata2.obs[['orig.ident']]='UC_batch2'
adata2.obs['patient']=['UC_P4' if int(i) <=26 else 'UC_P3' for i in adata2.obs['fov'].tolist() ]
adata2.obs['fov_cell_id']= [str(i)+'_'+str(j) for i,j in zip(adata2.obs['fov'].tolist(), adata2.obs['cell_id'].tolist())]


In [ ]:
adata2.var['neg_probe']=[True if 'Negative' in i else False for i in adata2.var.index.tolist()]
adata2.var['control_prob']=[True if 'SystemContro' in i else False for i in adata2.var.index.tolist()]
adata2.var

# merge qc

In [ ]:
adata_cosmx=sc.concat([adata, adata2])

In [ ]:
adata_cosmx.var['neg_probe']=[True if 'Negative' in i else False for i in adata_cosmx.var.index.tolist()]
adata_cosmx.var['control_probe']=[True if 'SystemControl' in i else False for i in adata_cosmx.var.index.tolist()]
adata_cosmx.var

In [ ]:
adata_cosmx=adata_cosmx[:, ~adata_cosmx.var["neg_probe"]]
adata_cosmx=adata_cosmx[:, ~adata_cosmx.var["control_probe"]]

In [ ]:
sc.pp.calculate_qc_metrics(
    adata_cosmx, inplace=True, log1p=True
)

In [ ]:

sc.pl.violin(
    adata_cosmx,
    ["n_genes_by_counts", "total_counts",'Area'],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
# only remove cells that have too little gene counts
sc.pp.filter_cells(adata_cosmx, min_counts=400)
sc.pp.filter_cells(adata_cosmx, min_genes=200)
sc.pp.filter_cells(adata_cosmx, max_counts=11000)
sc.pp.filter_cells(adata_cosmx, max_genes=7000)

In [ ]:
adata_cosmx=adata_cosmx[adata_cosmx.obs['Area']<=30000]

In [ ]:
sc.pl.violin(
    adata_cosmx,
    ["n_genes_by_counts", "total_counts",'Area'],
    jitter=0.4,
    multi_panel=True,
)

# individual filter

In [ ]:
adata=adata[:, ~adata.var["neg_probe"]]
adata=adata[:, ~adata.var["control_prob"]]

adata2=adata2[:, ~adata2.var["neg_probe"]]
adata2=adata2[:, ~adata2.var["control_prob"]]

In [ ]:
sc.pp.calculate_qc_metrics(
    adata, inplace=True, log1p=True
)

In [ ]:
sc.pp.calculate_qc_metrics(
    adata2, inplace=True, log1p=True
)

In [ ]:
# only remove cells that have too little gene counts
sc.pp.filter_cells(adata, min_counts=400)
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_cells(adata, max_counts=11000)
sc.pp.filter_cells(adata, max_genes=7000)adata1_slide3_1k

In [ ]:
# only remove cells that have too little gene counts
sc.pp.filter_cells(adata2, min_counts=400)
sc.pp.filter_cells(adata2, min_genes=200)
sc.pp.filter_cells(adata2, max_counts=11000)
sc.pp.filter_cells(adata2, max_genes=7000)

In [ ]:
adata=adata[adata.obs['Area']<=30000]

In [ ]:
adata2=adata2[adata2.obs['Area']<=30000]

In [ ]:
adata_cosmx.write_h5ad('/path/to/cosmx_data/uc_1k_6k_wtx/whole_trans/Processed_merged/v7/h5ad_obj/qc_wtx_v7_merged_no_norm.h5ad')

In [ ]:
adata_cosmx=sc.read_h5ad('/path/to/cosmx_data/uc_1k_6k_wtx/whole_trans/Processed_merged/v7/h5ad_obj/qc_wtx_v7_merged_no_norm.h5ad')

In [ ]:
adata.write_h5ad('/path/to/cosmx_data/uc_1k_6k_wtx/whole_trans/Processed_UC_P1_UC_P2/v7/h5ad_obj/qc_wtx_v7_UC_batch1_no_norm.h5ad')
adata2.write_h5ad('/path/to/cosmx_data/uc_1k_6k_wtx/whole_trans/Processed_UC_P3_UC_P4/v7/h5ad_obj/qc_wtx_v7_UC_batch2_no_norm.h5ad')

# comparison with scrna

In [ ]:
adata_1k=sc.read_h5ad('/path/to/cosmx_data/uc_1k_6k_wtx/whole_trans/Processed_merged/v7/h5ad_obj/qc_wtx_v7_merged_no_norm.h5ad')

In [ ]:
scrna=sc.read_h5ad('/path/to/scrna/uc/cleaned_annoed_all_cell_types.h5ad')

In [ ]:
scrna.X=pd.DataFrame(scrna.raw.X.toarray())

In [ ]:
scrna.obs['Technology']='scRNA'

In [ ]:
scrna.obs=scrna.obs.drop(['orig.ident','sample','Patient','condition','label'],axis=1)

In [ ]:
adata_cosmx.X=pd.DataFrame(adata_cosmx.X)

In [ ]:
adata_raw = adata_cosmx.copy()

In [ ]:
adata_raw.obs['Technology']='CosMx V7'

In [ ]:
adata_raw.obs=adata_raw.obs[['Technology']]

In [ ]:
del adata_raw.uns
del adata_raw.obsm
del adata_raw.varm
del adata_raw.layers
del adata_raw.obsp

In [ ]:
adata_raw.var['features']=adata_raw.var.index.tolist()

In [ ]:
adata_raw.var=adata_raw.var[['features']]

In [ ]:
shared_genes=list(set(adata_raw.var.index.tolist()).intersection(set(scrna.var.index.tolist())))

In [ ]:
adata_raw=adata_raw[:,shared_genes]

In [ ]:
scrna=scrna[:,shared_genes]

In [ ]:
# Calculate nFeature (number of genes detected per cell)
adata_raw.obs['nFeature'] = (adata_raw.X > 0).sum(axis=1)
# Calculate nCount (total counts per cell)
adata_raw.obs['nCount'] = adata_raw.X.sum(axis=1)

In [ ]:
sc.pp.calculate_qc_metrics(scrna, inplace=True)

In [ ]:
scrna.obs["nFeature"] = scrna.obs["n_genes_by_counts"]
scrna.obs["nCount"]   = scrna.obs["total_counts"]

In [ ]:
obs=pd.concat([adata_raw.obs, scrna.obs])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Do NOT set any Seaborn theme — leave it in default matplotlib mode

fig, ax = plt.subplots(figsize=(5, 4))  # Set figure size only

# Violin plot using seaborn, but no style override
sns.violinplot(data=obs, x="Technology", y="nFeature", hue="Technology", legend=False, ax=ax)

# Axis labels and title
ax.set_xlabel("Technology", fontsize=16)
ax.set_ylabel("Unique Gene per Cell", fontsize=16)
ax.set_title("Unique Genes Comparison", fontsize=18)

# Custom x-tick labels
ax.set_xticks([0, 1])
ax.set_xticklabels([
    "CosMx\n" + r"$\mathregular{(n=107,326\ cells)}$",
    "scRNA\n" + r"$\mathregular{(n=442,725\ cells)}$"
], fontsize=14)

# Let matplotlib handle ticks naturally (no manual styling needed)
plt.tight_layout()
plt.show()


# continuous processing normalization + batch effect + umap

In [ ]:
slide=[]
for i,j in zip(adata_cosmx.obs['orig.ident'].tolist(),adata_cosmx.obs['fov'].tolist()):
    j= int(j)
    if i=='UC_batch1':
        if j <=9:
            slide.append(i+'_'+'slide1')
        elif j > 9 and j <= 32:
            slide.append(i+'_'+'slide2')
        elif j > 32 and j <= 44:
            slide.append(i+'_'+'slide3')
        else:
            slide.append(i+'_'+'slide4')
    elif i=='UC_batch2':
        if j <=15:
            slide.append(i+'_'+'slide1')
        elif j > 15 and j <= 26:
            slide.append(i+'_'+'slide2')
        elif j > 26 and j <= 43:
            slide.append(i+'_'+'slide3')
        else:
            slide.append(i+'_'+'slide4')

In [ ]:
adata_cosmx.obs['slide']=slide

In [ ]:
adata_cosmx.obs['slide'].value_counts()

In [ ]:
adata_cosmx.layers['raw']=adata_cosmx.X

In [ ]:
adata.layers['raw']=pd.DataFrame(adata.X.toarray())
adata2.layers['raw']=pd.DataFrame(adata2.X.toarray())

In [ ]:
sc.pp.normalize_total(adata_cosmx, inplace=True)
sc.pp.log1p(adata_cosmx)


In [ ]:
adata_cosmx_norm_count1 = adata_cosmx[adata_cosmx.obs['patient'].isin(['UC_P1','UC_P2'])].X
adata_cosmx_norm_count1

In [ ]:
adata_cosmx_norm_count2 = adata_cosmx[adata_cosmx.obs['patient'].isin(['UC_P3','UC_P4'])].X
adata_cosmx_norm_count2

In [ ]:
adata.X = adata_cosmx_norm_count1

In [ ]:
adata2.X = adata_cosmx_norm_count2

In [ ]:
adata_cosmx.write_h5ad('/path/to/cosmx_data/uc_1k_6k_wtx/whole_trans/Processed_merged/v7/h5ad_obj/qc_wtx_v7_merged_norm_umap.h5ad')

In [ ]:
adata.write_h5ad('/path/to/cosmx_data/uc_1k_6k_wtx/whole_trans/Processed_UC_P1_UC_P2/v7/h5ad_obj/qc_wtx_v7_UC_batch1_norm_leiden.h5ad')
adata2.write_h5ad('/path/to/cosmx_data/uc_1k_6k_wtx/whole_trans/Processed_UC_P3_UC_P4/v7/h5ad_obj/qc_wtx_v7_UC_batch2_norm_leiden.h5ad')

In [ ]:
adata_cosmx_norm_nolog1p = adata_cosmx.copy()
adata_cosmx_norm_nolog1p.X = adata_cosmx_norm_nolog1p.layers['raw']


In [ ]:
sc.pp.normalize_total(adata_cosmx_norm_nolog1p, inplace=True)

In [ ]:
adata_cosmx_norm_nolog1p_count = pd.DataFrame(adata_cosmx_norm_nolog1p.X)

In [ ]:
adata_cosmx_norm_nolog1p_count.columns = adata_cosmx_norm_nolog1p.var.index.tolist()
adata_cosmx_norm_nolog1p_count.index=adata_cosmx_norm_nolog1p.obs['cell'].tolist()

In [ ]:
adata_cosmx_norm_nolog1p_count.to_csv('/path/to/cosmx_data/uc_1k_6k_wtx/whole_trans/Processed_merged/v7/h5ad_obj/norm_counts.csv')

# anno

In [ ]:
aucell_norm_count = pd.read_csv('/path/to/cosmx_data/uc_1k_6k_wtx/whole_trans/Processed_merged/v7/aucell/norm_counts_aucell.csv')

In [ ]:
mapper= pd.read_csv('/path/to/scrna/uc/cell_type_name_mapper.csv')

In [ ]:
aucell_norm_count=aucell_norm_count.merge(mapper, how = 'left', left_on = 'label',right_on = 'cell_type_short')

In [ ]:
adata_cosmx.obs['aucell_cell_type_short'] = aucell_norm_count['cell_type_short'].tolist()
adata_cosmx.obs['aucell_cell_type_full'] = aucell_norm_count['cell_type'].tolist()
adata_cosmx.obs['aucell_cell_category_from_type'] = aucell_norm_count['cell_category'].tolist()
adata_cosmx.obs['aucell_cell_type_general'] = aucell_norm_count['cell_type_general'].tolist()

In [ ]:
adata_cosmx1 = adata_cosmx[adata_cosmx.obs['patient'].isin(['UC_P1','UC_P2'])]
adata_cosmx1_anno = adata_cosmx1.obs[['aucell_cell_type_short','aucell_cell_type_full','aucell_cell_category_from_type','aucell_cell_type_general']]

In [ ]:
adata_cosmx2 = adata_cosmx[adata_cosmx.obs['patient'].isin(['UC_P3','UC_P4'])]
adata_cosmx2_anno = adata_cosmx2.obs[['aucell_cell_type_short','aucell_cell_type_full','aucell_cell_category_from_type','aucell_cell_type_general']]

In [ ]:
adata.obs = adata.obs.merge(adata_cosmx1_anno, how = 'left', left_index=True,right_index=True)

In [ ]:
adata2.obs = adata2.obs.merge(adata_cosmx2_anno, how = 'left', left_index=True,right_index=True)

In [ ]:
adata_cosmx.write_h5ad('/path/to/cosmx_data/uc_1k_6k_wtx/whole_trans/Processed_merged/v7/h5ad_obj/qc_wtx_v7_merged_anno.h5ad')

In [ ]:
adata.write_h5ad('/path/to/cosmx_data/uc_1k_6k_wtx/whole_trans/Processed_UC_P1_UC_P2/v7/h5ad_obj/qc_wtx_v7_UC_P1_UC_P2_anno.h5ad')

In [ ]:
adata2.write_h5ad('/path/to/cosmx_data/uc_1k_6k_wtx/whole_trans/Processed_UC_P3_UC_P4/v7/h5ad_obj/qc_wtx_v7_UC_P3_UC_P4_anno.h5ad')

# plotting

In [ ]:
# Create a color palette
coordinates = adata.obsm['spatial_fov']

category_column = 'aucell_cell_category_from_type'
categories = adata.obs[category_column].astype('category')

palette = sns.color_palette('tab20', len(categories.cat.categories))
colors = categories.cat.codes.map(dict(enumerate(palette)))

# Plotting
plt.figure(figsize=(9, 10))
scatter = plt.scatter(coordinates[:, 0], coordinates[:, 1], c=colors, s=0.5)  # Adjust `s` for marker size
plt.title("UC_P1_UC_P2 Spatial Plot for All FOVs")
plt.xlabel("X coordinate")
plt.ylabel("Y coordinate")

# Create a custom legend
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=palette[i], markersize=10) 
           for i in range(len(categories.cat.categories))]
plt.legend(handles, categories.cat.categories, title=category_column, bbox_to_anchor=(1.05, 1), loc='upper left')

plt.show()

In [ ]:
# Create a color palette
coordinates = adata2.obsm['spatial_fov']

category_column = 'aucell_cell_category_from_type'
categories = adata2.obs[category_column].astype('category')

palette = sns.color_palette('tab20', len(categories.cat.categories))
colors = categories.cat.codes.map(dict(enumerate(palette)))

# Plotting
plt.figure(figsize=(9, 10))
scatter = plt.scatter(coordinates[:, 0], coordinates[:, 1], c=colors, s=0.5)  # Adjust `s` for marker size
plt.title("UC_P3_UC_P4 Spatial Plot for All FOVs")
plt.xlabel("X coordinate")
plt.ylabel("Y coordinate")

# Create a custom legend
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=palette[i], markersize=10) 
           for i in range(len(categories.cat.categories))]
plt.legend(handles, categories.cat.categories, title=category_column, bbox_to_anchor=(1.05, 1), loc='upper left')

plt.show()

In [ ]:
# Create a color palette
adata1_slide3_1k = adata[adata.obs['slide']=='UC_batch1_slide3']
coordinates = adata1_slide3_1k.obsm['spatial_fov']

category_column = 'aucell_cell_category_from_type'
categories = adata1_slide3_1k.obs[category_column].astype('category')

palette = sns.color_palette('tab20', len(categories.cat.categories))
colors = categories.cat.codes.map(dict(enumerate(palette)))

# Plotting
plt.figure(figsize=(8, 10))
scatter = plt.scatter(coordinates[:, 0], coordinates[:, 1], c=colors, s=2)  # Adjust `s` for marker size
plt.title("WTX V7 UC_P1 Left Tissue Spatial Plot for AUCell")
plt.xlabel("X coordinate")
plt.ylabel("Y coordinate")

# Remove gridlines
plt.grid(False)  # Disable gridlines

# Create a custom legend
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=palette[i], markersize=10) 
           for i in range(len(categories.cat.categories))]
plt.legend(handles, categories.cat.categories, title=category_column, bbox_to_anchor=(1.05, 1), loc='upper left')

plt.show()


In [ ]:
# Create a color palette
adata1_slide3_1k = adata[adata.obs['slide']=='UC_batch1_slide4']
coordinates = adata1_slide3_1k.obsm['spatial_fov']

category_column = 'aucell_cell_category_from_type'
categories = adata1_slide3_1k.obs[category_column].astype('category')

palette = sns.color_palette('tab20', len(categories.cat.categories))
colors = categories.cat.codes.map(dict(enumerate(palette)))

# Plotting
plt.figure(figsize=(8, 10))
scatter = plt.scatter(coordinates[:, 0], coordinates[:, 1], c=colors, s=2)  # Adjust `s` for marker size
plt.title("WTX V7 UC_P1 Left Tissue Spatial Plot for AUCell")
plt.xlabel("X coordinate")
plt.ylabel("Y coordinate")

# Remove gridlines
plt.grid(False)  # Disable gridlines

# Create a custom legend
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=palette[i], markersize=10) 
           for i in range(len(categories.cat.categories))]
plt.legend(handles, categories.cat.categories, title=category_column, bbox_to_anchor=(1.05, 1), loc='upper left')

plt.show()


# diversity comparison

In [ ]:
adata_wtx=sc.read_h5ad('/path/to/cosmx_data/uc_1k_6k_wtx/whole_trans/Processed_merged/v7/h5ad_obj/qc_wtx_v7_merged_anno.h5ad')

In [ ]:
adata_6k=sc.read_h5ad('/path/to/cosmx_data/uc_1k_6k_wtx/6k/Processed_merged_upd/h5ad_obj/qc_6k_merged_log1p_6175genes.h5ad')

In [ ]:
adata_6k_cus=sc.read_h5ad('/path/to/cosmx_data/uc_1k_6k_wtx/6k/Processed_merged_upd/h5ad_obj/qc_6k_merged_log1p_6375genes.h5ad')

In [ ]:
df_collapsed_updated.to_csv('/path/to/scrna/cell_markers/custom_cosmx_markers.csv')

In [ ]:
adata_1k=sc.read_h5ad('/path/to/cosmx_data/uc_1k_6k_wtx/1k/Processed_merged/h5ad_obj/imp_ywl_anno.h5ad')

In [ ]:
qc_wtx =adata_wtx.obs[['n_genes_by_counts', 'total_counts']]
qc_wtx['plex']='wtx'

In [ ]:
qc_1k =adata_1k.obs[['n_genes_by_counts', 'total_counts']]
qc_1k['plex']='1k'

In [ ]:
qc_6k =adata_6k.obs[['n_genes_by_counts', 'total_counts']]
qc_6k['plex']='6k'

In [ ]:
qc_merge=pd.concat([pd.concat([qc_wtx,qc_6k],ignore_index=True),qc_1k],ignore_index=True)

In [ ]:
qc_merge.columns = ['Diversity','Total Count','Plex']

In [ ]:
# List of variables to plot
variables = ['Total Count','Diversity']

# Create subplots for each variable with separate y-axes
fig, ax = plt.subplots(1, 2, figsize=(8, 3))  # Reduced figure size

# Using Seaborn's 'tab10' palette to match default color scheme
colors = sns.color_palette('tab10')  
bar_colors = [colors[0], colors[2], 'pink']  # Specify your custom colors

# Plot KDE for each variable
for i, var in enumerate(variables):
    sns.kdeplot(data=qc_merge, x=var, hue="Plex", palette=bar_colors, 
                common_norm=False, shade=True, alpha=0.4, ax=ax[i])
    ax[i].set_title('', fontsize=14)  # Larger font for title
    ax[i].set_ylabel("Proportion", fontsize=12)  # Larger font for y-label
    ax[i].set_xlabel("Unique Gene per Cell", fontsize=12)  # Removing x-label and adjusting font size if needed
    ax[i].tick_params(axis='both', labelsize=10)  # Larger font for ticks
    legend = ax[i].get_legend()
    if legend:
        legend.set_title("All Genes")
# Adjust layout to avoid overlap
plt.tight_layout()

plt.show()

# umi

In [ ]:
def _total_counts_per_cell(adata):
    """Robust total counts per cell from .X (works for dense or sparse)."""
    X = adata.X
    s = X.sum(axis=1)
    # s can be matrix type; convert to flat ndarray
    if hasattr(s, "A1"):
        return s.A1
    return np.asarray(s).ravel()

def _n_detected_genes_per_cell(adata):
    """# detected genes per cell (nonzero entries). Works for sparse/dense."""
    X = adata.X
    if hasattr(X, "getnnz"):  # sparse
        return X.getnnz(axis=1)
    return np.count_nonzero(np.asarray(X), axis=1)

def compute_umis_per_detected_gene(adata, eps=1e-8):
    total = _total_counts_per_cell(adata).astype(float)
    ngen = _n_detected_genes_per_cell(adata).astype(float)
    return total / (ngen + eps)

def subset_to_shared_genes(adatas_dict):
    """Return new dict of adatas subset to shared var_names intersection."""
    shared = None
    for ad in adatas_dict.values():
        genes = set(ad.var_names)
        shared = genes if shared is None else (shared & genes)
    shared = sorted(shared)
    return {k: v[:, shared].copy() for k, v in adatas_dict.items()}

def plot_density(metrics_dict, title, log1p=True, max_cells=200_000):
    """
    metrics_dict: {label: 1D array}
    Plots overlaid density curves using hist-based density (no seaborn).
    """
    plt.figure(figsize=(5.2, 3.6))
    for label, vals in metrics_dict.items():
        vals = np.asarray(vals)
        vals = vals[np.isfinite(vals)]
        if max_cells is not None and vals.size > max_cells:
            rng = np.random.default_rng(0)
            vals = rng.choice(vals, size=max_cells, replace=False)

        x = np.log1p(vals) if log1p else vals

        # density via histogram (stable and fast)
        hist, edges = np.histogram(x, bins=200, density=True)
        centers = 0.5 * (edges[:-1] + edges[1:])
        plt.plot(centers, hist, label=label)

    plt.title(title)
    plt.ylabel("Density")
    plt.xlabel("log1p(UMIs per detected gene per cell)" if log1p else "UMIs per detected gene per cell")
    plt.legend(title="Plex")
    plt.tight_layout()
    plt.show()

def plot_violin(metrics_dict, title, log1p=True, max_cells=200_000):
    """Violin plot (matplotlib only)."""
    labels = list(metrics_dict.keys())
    data = []
    rng = np.random.default_rng(0)

    for label in labels:
        vals = np.asarray(metrics_dict[label])
        vals = vals[np.isfinite(vals)]
        if max_cells is not None and vals.size > max_cells:
            vals = rng.choice(vals, size=max_cells, replace=False)
        vals = np.log1p(vals) if log1p else vals
        data.append(vals)

    plt.figure(figsize=(6.0, 3.8))
    vp = plt.violinplot(data, showmeans=False, showmedians=True, showextrema=False)
    # keep default styling; just set x ticks/labels
    plt.xticks(np.arange(1, len(labels) + 1), labels)
    plt.title(title)
    plt.ylabel("log1p(UMIs per Detected Gene per cell)" if log1p else "UMIs per Detected Gene per Cell")
    plt.tight_layout()
    plt.show()


In [ ]:
adatas = {
    "WTX": adata_cosmx,
    "6K": adata_6k,
    "1K": adata_1k
}


In [ ]:
metrics_all = {k: compute_umis_per_detected_gene(v) for k, v in adatas.items()}
plot_density(metrics_all, title="UMIs per Detected Gene per Cell", log1p=True)
plot_violin(metrics_all, title="UMIs per Detected Gene per Cell", log1p=True)


In [ ]:
import numpy as np

def umis_per_gene_raw(adata):
    X = adata.layers["counts"] if "counts" in adata.layers else adata.X
    total = X.sum(axis=1)
    if hasattr(total, "A1"):
        total = total.A1
    detected = X.getnnz(axis=1) if hasattr(X, "getnnz") else (X > 0).sum(axis=1)
    return total / detected

for name, ad in [("WTX", adata_cosmx), ("6k", adata_6k), ("1k", adata_1k)]:
    vals = umis_per_gene_raw(ad)
    print(
        name,
        "median =", np.median(vals),
        "mean =", np.mean(vals)
    )


In [ ]:
def gene_range(adata, q_low=5, q_high=95):
    X = adata.layers["counts"] if "counts" in adata.layers else adata.X
    X = X.toarray() if hasattr(X, "toarray") else X

    lo = np.percentile(X, q_low, axis=0)
    hi = np.percentile(X, q_high, axis=0)

    return hi - lo

rng_wtx = gene_range(adata_cosmx[:, shared])
rng_6k  = gene_range(adata_6k[:, shared])
rng_1k  = gene_range(adata_1k[:, shared])


In [ ]:
plt.figure(figsize=(5,4))
plt.violinplot([rng_wtx, rng_6k, rng_1k], showmedians=True)
plt.xticks([1,2,3], ["WTX", "6K", "1K"])
plt.ylabel("95–5% Expression Range per Gene (UMIs)")
plt.title("Gene-level Expression Dynamic Range")
plt.tight_layout()
plt.show()


# imp neg gene validation 

In [ ]:
ywlb_UC_P1 = pd.read_csv(
    "/path/to/imputation_model/CosMx/results/Jenny_cosmx_geneimputation/results/1K_imputation/20250215/1K_pred_UC_batch1_b.csv"
)

In [ ]:
ywlb_UC_P1_lg = pd.read_csv(
    "/path/to/imputation_model/CosMx/results/Jenny_cosmx_geneimputation/results/1K_imputation/20250215/1K_pred_UC_batch1_b_log1p.csv"
)

In [ ]:
sub = adata_1k[adata_1k.obs['patient'].isin(['UC_P1','UC_P2'])]

In [ ]:
sub.obs['v7_ywl_b_imp_cell_category_from_type'].value_counts()

In [ ]:
sub=sub[sub.obs['v7_ywl_b_imp_cell_category_from_type'].isin(['Epithelial','T','Myeloid','Plasma','Connective'])]

In [ ]:
from anndata import AnnData

In [ ]:
ywlb_UC_P1_lg_anno = ywlb_UC_P1_lg.merge(sub.obs[['v7_ywl_b_imp_cell_category_from_type']], how = 'right',right_index=True, left_on = 'old_index')

In [ ]:
ywlb_UC_P1_lg_anno.index = ywlb_UC_P1_lg_anno['old_index'].tolist()

In [ ]:
gene_cols = ywlb_UC_P1_lg_anno.columns.tolist()[1:-1]

X = ywlb_UC_P1_lg_anno[gene_cols].to_numpy()
adata = AnnData(X)
adata.var_names = gene_cols
adata

In [ ]:
adata.obs["cell_type"] =ywlb_UC_P1_lg_anno['v7_ywl_b_imp_cell_category_from_type'].tolist()

In [ ]:
adata.obs["lineage3"] = adata.obs["cell_type"].replace({
    "Myeloid": "Immune",
    "T": "Immune",
    "Plasma": "Immune",
    "Epithelial": "Epithelial",
    "Connective": "Connective",
}).astype("category")

# sanity check
print(adata.obs["lineage3"].value_counts())


In [ ]:
sc.tl.rank_genes_groups(
    adata,
    groupby="lineage3",
    groups=["Immune"],
    reference="rest",
    method="wilcoxon",   # robust default
    n_genes=500
)

de = sc.get.rank_genes_groups_df(adata, group="Immune")
de.head()

In [ ]:
# compute per-lineage mean on the same matrix you DE'd on (adata.X)
lineage_means = {}
for g in ["Immune", "Epithelial", "Connective"]:
    lineage_means[g] = np.asarray(adata[adata.obs["lineage3"] == g].X).mean(axis=0)

mean_immune = lineage_means["Immune"]
mean_epi    = lineage_means["Epithelial"]
mean_fib    = lineage_means["Connective"]

gene_to_idx = {g:i for i,g in enumerate(adata.var_names)}

def get_means(genes):
    idx = [gene_to_idx[g] for g in genes]
    return (mean_immune[idx], mean_epi[idx], mean_fib[idx])

# candidate genes from DE table
cand = de["names"].tolist()

mi, me, mf = get_means(cand)

de["mean_immune"] = mi
de["mean_epi"]    = me
de["mean_fib"]    = mf



In [ ]:
de[:50]['names'].tolist()

In [ ]:
sc.tl.rank_genes_groups(
    adata,
    groupby="lineage3",
    groups=["Connective"],
    reference="rest",
    method="wilcoxon",   # robust default
    n_genes=500
)


In [ ]:
de2 = sc.get.rank_genes_groups_df(adata, group="Connective")
de2.head()

In [ ]:
# compute per-lineage mean on the same matrix you DE'd on (adata.X)
lineage_means = {}
for g in ["Immune", "Epithelial", "Connective"]:
    lineage_means[g] = np.asarray(adata[adata.obs["lineage3"] == g].X).mean(axis=0)

mean_immune = lineage_means["Immune"]
mean_epi    = lineage_means["Epithelial"]
mean_fib    = lineage_means["Connective"]

gene_to_idx = {g:i for i,g in enumerate(adata.var_names)}

def get_means(genes):
    idx = [gene_to_idx[g] for g in genes]
    return (mean_immune[idx], mean_epi[idx], mean_fib[idx])

# candidate genes from DE table
cand = de2["names"].tolist()

mi, me, mf = get_means(cand)

de2["mean_immune"] = mi
de2["mean_epi"]    = me
de2["mean_fib"]    = mf



In [ ]:
negative_control_genes_imp=[
     "MUC1", "MUC2", "VIL1", "TFF1", "MUC13",
            "LCP1", 
        "LAPTM5",      # lysosomal protein, hematopoietic
        "SAMHD1",      # immune antiviral / myeloid & lymphoid
    "LSP1",        # pan-leukocyte cytoskeleton adaptor
    "CORO1A",      # actin regulation, immune-specific"AEBP1",
    "GREM1",
    "FSTL1",
    "SOD3",
    "TNC",
    "TGM2",
]
negative_control_genes_imp.append('v7_ywl_b_imp_cell_category_from_type')

In [ ]:
df = ywlb_UC_P1_lg_anno[negative_control_genes_imp]

In [ ]:
df.columns = [    # Epithelial
     "MUC1", "MUC2", "VIL1", "TFF1", "MUC13",
            "LCP1", 
        "LAPTM5",      # lysosomal protein, hematopoietic
        "SAMHD1",      # immune antiviral / myeloid & lymphoid
    "LSP1",        # pan-leukocyte cytoskeleton adaptor
    "CORO1A",      # actin regulation, immune-specific"AEBP1",
    "GREM1",
    "FSTL1",
    "SOD3",
    "TNC",
    "TGM2",
    'cell_type']

In [ ]:
genes = df.columns[:-1]          # all gene columns
cell_type_col = df.columns[-1]   # last column

summary = []

for ct, g in df.groupby(cell_type_col):
    for gene in genes:
        vals = g[gene].values

        pct_expr = np.mean(vals > 0) * 100
        mean_expr = vals[vals > 0].mean() if np.any(vals > 0) else 0

        summary.append({
            "cell_type": ct,
            "gene": gene,
            "pct_expr": pct_expr,
            "mean_expr": mean_expr
        })

dot_df = pd.DataFrame(summary)

In [ ]:

cell_type_order = [
    "Epithelial",
    "Myeloid",
    "T",
    "Plasma",
    "Connective"
]
dot_df["gene"] = pd.Categorical(dot_df["gene"], categories=negative_control_genes_imp[:-1], ordered=True)
dot_df["cell_type"] = pd.Categorical(
    dot_df["cell_type"],
    categories=cell_type_order,
    ordered=True
)

In [ ]:
cell_type_col = df.columns[-1]  # or "cell_type"

df2 = df.copy()
df2["cell_type_collapsed"] = df2[cell_type_col].replace({
    "Myeloid": "Immune",
    "T": "Immune",
    "Plasma": "Immune"
})


In [ ]:
import numpy as np
import pandas as pd

genes = df2.columns[:-2]  # all gene columns (exclude original cell type + collapsed)
ct_col = "cell_type_collapsed"

summary = []
for ct, g in df2.groupby(ct_col):
    for gene in genes:
        vals = g[gene].values
        expr_thresh = 0.25  # try 0.1, 0.5, 1.0 and see what separates best

        pct_expr = np.mean(vals > expr_thresh) * 100
        mean_expr = vals[vals > expr_thresh].mean() if np.any(vals > expr_thresh) else 0
        summary.append({"cell_type": ct, "gene": gene, "pct_expr": pct_expr, "mean_expr": mean_expr})

dot_df = pd.DataFrame(summary)


In [ ]:
# Desired order
cell_type_order = ["Connective", "Immune", "Epithelial"]
dot_df["cell_type"] = pd.Categorical(dot_df["cell_type"], categories=cell_type_order, ordered=True)

# Compact spacing (default spacing=1; reduce to tighten)
spacing = 0.6

y_map = {ct: i * spacing for i, ct in enumerate(cell_type_order)}
dot_df["y_pos"] = dot_df["cell_type"].map(y_map)

# Make gene a categorical (optional but nice)
dot_df["gene"] = pd.Categorical(dot_df["gene"], categories=negative_control_genes_imp[:-1], ordered=True)

# Numeric x positions for plotting
x_map = {g: i for i, g in enumerate(ordered_genes)}
dot_df["x_pos"] = dot_df["gene"].map(x_map)


import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl
from matplotlib.lines import Line2D

fig, ax = plt.subplots(figsize=(10, 2.4))  # compact height

sc = ax.scatter(
    x=dot_df["gene"],
    y=dot_df["y_pos"],
    s=dot_df["pct_expr"] * 2.5,              # scale size manually
    c=dot_df["mean_expr"],
    cmap="Reds",
    edgecolor="black",
    linewidth=0.2
)

# ---- y-axis formatting ----
ax.set_yticks(list(y_map.values()))
ax.set_yticklabels(cell_type_order)
ax.set_ylim(-0.3, max(y_map.values()) + 0.3)  # no clipping

# ---- x-axis formatting ----
ax.set_xticks(range(len(dot_df["gene"].cat.categories)))
ax.set_xticklabels(dot_df["gene"].cat.categories, rotation=45, ha="right")

ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title("Negative Controls (Imputed Lineage Genes)")

# ---- colorbar ----
norm = mpl.colors.Normalize(
    vmin=dot_df["mean_expr"].min(),
    vmax=dot_df["mean_expr"].max()
)
cbar = fig.colorbar(
    mpl.cm.ScalarMappable(norm=norm, cmap="Reds"),
    ax=ax,
    fraction=0.035,
    pad=0.02
)
cbar.set_label("Mean expression")

size_levels = [10, 30, 60, 90]  # % expressed
size_scale = 2.5               # must match your scatter scaling

size_handles = [
    Line2D(
        [], [], 
        marker='o',
        linestyle='',
        markersize=(lvl * size_scale) ** 0.5,  # sqrt because marker size is area
        markerfacecolor='gray',
        markeredgecolor='black',
        label=f"{lvl}%"
    )
    for lvl in size_levels 
]

size_legend = ax.legend(
    handles=size_handles,
    title="% cells expressed",
    frameon=False,
    loc="upper left",
    bbox_to_anchor=(1.1, 0.97)
)

ax.add_artist(size_legend)  # important so it doesn't get overwritten
plt.tight_layout(rect=[0, 0, 0.7, 1])
fig.subplots_adjust(right=0.8)  # try 0.75–0.82
plt.show()
